# ⚙️ CareerLens AI — Phase 2: PDF Parsing, Section Segmentation & Zero-Shot NLP

**Pipeline Overview**:
1. PyMuPDF CV-based bounding box layout analysis and text extraction.
2. Text cleaning, null character stripping, and regex normalization.
3. Heuristic section segmentation into functional blocks.
4. Zero-Shot Named Entity Recognition with GLiNER (`urchade/gliner_multi-v2.1`).
5. Standardized Skill Ontology normalization.
6. Generation and verification of `data/processed_resumes.csv`.


In [1]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import re
import json
import pandas as pd
import numpy as np
import pymupdf as fitz
from lib.resume.ontology import SKILL_ONTOLOGY, normalize_skill

print("Preprocessing modules successfully loaded.")

Preprocessing modules successfully loaded.


## 1. PyMuPDF Layout Extraction & Text Cleaning

In [2]:
from lib.resume.segmenter import clean_text

sample_raw = "John Doe\x00   Software Engineer\n\n\nSkills:\nPython, SQL, React"
cleaned = clean_text(sample_raw)
print("Raw input snippet:     ", repr(sample_raw))
print("Cleaned text output:   ", repr(cleaned))

Raw input snippet:      'John Doe\x00   Software Engineer\n\n\nSkills:\nPython, SQL, React'
Cleaned text output:    'John Doe Software Engineer\n\nSkills:\nPython, SQL, React'


## 2. Section Segmentation Heuristics

In [3]:
from lib.resume.segmenter import split_into_sections

sample_cv_text = """John Doe
john.doe@email.com

SKILLS
Python, SQL, React, Next.js, Docker, PyTorch

EXPERIENCE
Software Engineer at Tech Corp
Developed scalable microservices and APIs.

EDUCATION
Bachelor of Science in Computer Science, MIT

PROJECTS
CareerLens AI
Built automated resume parsing and intelligence platform.
"""

sections = split_into_sections(sample_cv_text)
print("Segmented Section Keys:", list(sections.keys()))
for k, v in sections.items():
    if v.strip():
        print(f"\n--- [{k}] ---\n{v.strip()}")

Segmented Section Keys: ['SKILLS', 'EXPERIENCE', 'EDUCATION', 'PROJECTS', 'CERTIFICATIONS', 'UNCLASSIFIED']

--- [SKILLS] ---
Python, SQL, React, Next.js, Docker, PyTorch

--- [EXPERIENCE] ---
Software Engineer at Tech Corp
Developed scalable microservices and APIs.

--- [EDUCATION] ---
Bachelor of Science in Computer Science, MIT

--- [PROJECTS] ---
CareerLens AI
Built automated resume parsing and intelligence platform.

--- [UNCLASSIFIED] ---
John Doe
john.doe@email.com


## 3. Skill Ontology and Normalization Dictionary

In [4]:
print(f"Total Standardized Skills in Ontology: {len(SKILL_ONTOLOGY)}")
sample_mappings = list(SKILL_ONTOLOGY.items())[:8]
for raw, canonical in sample_mappings:
    print(f"  '{raw}' -> '{canonical}'")

Total Standardized Skills in Ontology: 117
  'python' -> 'Python'
  'py' -> 'Python'
  'python3' -> 'Python'
  'javascript' -> 'JavaScript'
  'js' -> 'JavaScript'
  'typescript' -> 'TypeScript'
  'ts' -> 'TypeScript'
  'java' -> 'Java'


## 4. Verification of Canonical Processed Dataset

In [5]:
proc_path = '../data/processed_resumes.csv' if os.path.exists('../data/processed_resumes.csv') else 'nlp_processed_resumes.csv'
df = pd.read_csv(proc_path)

print(f"Processed Resumes Count: {len(df)}")
print("Schema Information:")
print(df.info())
display(df.head(3))

Processed Resumes Count: 2466
Schema Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2466 entries, 0 to 2465
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   filename         2466 non-null   object
 1   category         2466 non-null   object
 2   raw_clean_text   2466 non-null   object
 3   lemmatized_text  2466 non-null   object
dtypes: object(4)
memory usage: 77.2+ KB
None


,filename,category,raw_clean_text,lemmatized_text
0,10554236.pdf,ACCOUNTANT,ACCOUNTANT Summary Financial Accountant specia...,accountant summary financial accountant specia...
1,10674770.pdf,ACCOUNTANT,STAFF ACCOUNTANT Summary Highly analytical and...,staff accountant summary highly analytical det...
2,11163645.pdf,ACCOUNTANT,ACCOUNTANT Professional Summary To obtain a po...,accountant professional summary obtain positio...


In [6]:
# Verify that no empty or corrupted rows exist
null_counts = df.isnull().sum()
print("Null Values Check:")
print(null_counts)
assert null_counts['raw_clean_text'] == 0, "Error: Missing raw_clean_text found!"
print("Dataset verification passed successfully.")

Null Values Check:
filename           0
category           0
raw_clean_text     0
lemmatized_text    0
dtype: int64
Dataset verification passed successfully.
